# Chapter 3 (part 2) — Multiple Testing, ANOVA, Chi-Square, Bandits, and Power

Covers the rest of Ch 3 of *Practical Statistics for Data Scientists*: Multiple Testing → Degrees of Freedom → ANOVA → Chi-Square Test → Multi-Arm Bandit Algorithm → Power and Sample Size, applied to the Ames Housing dataset.

Several sections deliberately reuse the groups and results from `05_ab-testing-and-t-tests.ipynb` (the Average vs. Above_Average quality comparison) so you can see how these new tools relate to the t-test you already ran. Ask for a hint if you get stuck; per the repo's review protocol, the first couple of asks per question get a nudge, not the answer.

Primary tools: pandas / numpy for data wrangling, scipy.stats for the formal tests, statsmodels for ANOVA tables and power calculations.


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

ames = pd.read_csv('../data/ames_housing.csv')
ames.shape


## 1. Multiple Testing

Suppose you flip 20 fair coins, 100 times each, and ask "is this coin biased?" with `alpha = 0.05` for each one. Even though every coin is perfectly fair, you'd *expect* about 20 x 0.05 = 1 of those 20 tests to come back "significant" just from chance — that's what a 5% false-positive rate means when you take 20 independent shots at it.

Now translate that into a data-science habit that feels totally innocent: you have 28 neighborhoods in Ames, and you test *each one* to see if its average Sale_Price differs from the overall Ames average. That's 28 independent tests. If neighborhood had **zero** real effect on price, you'd still expect roughly 28 x 0.05 ≈ 1.4 of them to look "significant" at alpha = 0.05 purely by chance — and it's tempting to write a headline about whichever one popped.

1. For each neighborhood, test whether that neighborhood's mean `Sale_Price` differs from the rest of Ames (pick either a one-sample t-test against the overall mean, or a two-sample t-test against "everyone else" — your call, just be consistent). Count how many neighborhoods come back with p < 0.05.
2. Apply a Bonferroni correction — divide alpha by the number of tests you ran — and recount how many neighborhoods still clear that stricter bar.
3. In bullet points: how many "discoveries" evaporated after correction? Why would this matter if you were, say, screening all 28 neighborhoods to find "the one that's really different" rather than testing one neighborhood you already suspected going in?


**Your notes (bullet points):**

- 

## 2. Degrees of Freedom

Take 3 numbers: 4, 9, and 5. Their mean is 6. Now imagine I only tell you two facts: two of the three values are 4 and 9, and the mean of all three is 6. You can solve for the third value exactly — it has to be 5, there's no other option. Only 2 of the 3 values were actually "free to vary" once the mean was pinned down; the third was forced along for the ride. That's degrees of freedom: with n data points and one quantity (the mean) already estimated from them, you have n - 1 values that were free to vary. It's the same reason sample variance divides by (n - 1) instead of n — you "spent" one degree of freedom estimating the mean before you could estimate the spread around it.

This matters for the t-test you ran in notebook 05: the shape of the t-distribution used to compute your p-value depends on degrees of freedom, and the two flavors of t-test (equal-variance vs. Welch's) don't compute it the same way.

1. Recreate the two groups from notebook 05 (`Overall_Qual == "Average"` vs. `"Above_Average"`). For the classic equal-variance t-test, degrees of freedom is simply `df = n1 + n2 - 2`. Compute that by hand.
2. Re-run `scipy.stats.ttest_ind(..., equal_var=False)` (Welch's t-test) on the same two groups and inspect `res.df`. It won't match your answer from step 1, and it probably won't even be a whole number. Look up (or reason through) the Welch-Satterthwaite formula it's using — why does unequal variance between the two groups change how much "effective" information you have?


In [ ]:
# Recreate the Average vs. Above_Average Overall_Qual groups from notebook 05


**Your notes (bullet points):**

- 

## 3. ANOVA (F-Statistic, Two-Way ANOVA)

With 2 groups, one t-test settles the question. But Ames actually has 10 `Overall_Qual` tiers (Very_Poor through Very_Excellent). Comparing all of them pairwise would mean 45 separate t-tests — and you just saw in section 1 what running that many tests does to your false-positive rate.

ANOVA sidesteps the problem by asking one question instead of 45: is the *variance between the group means* large compared to the *variance within each group* (i.e., the ordinary noise you'd see even within a single quality tier)? That ratio is the **F-statistic**. Concretely: if 3 tiny groups had values `{2, 4}`, `{6, 8}`, `{10, 12}`, the within-group spread is small (each pair differs by only 2) while the between-group spread is huge (means of 3, 7, 11) — that combination produces a large F and a tiny p-value, i.e. "these group means are definitely not all the same."

1. Run a one-way ANOVA across **all** `Overall_Qual` tiers (not just two) predicting `Sale_Price` — `scipy.stats.f_oneway` or `statsmodels.formula.api.ols` + `anova_lm` both work. Report the F-statistic and p-value. What question is this answering that a single pairwise t-test (like the one in notebook 05) can't?
2. Now run a **two-way ANOVA**: add a second categorical factor, `Central_Air` (Y/N), alongside `Overall_Qual`, fitting something like `Sale_Price ~ C(Overall_Qual) + C(Central_Air) + C(Overall_Qual):C(Central_Air)` with `statsmodels`. Look at the ANOVA table — does quality, central air, or their *interaction* explain a meaningful share of the variance in price?


**Your notes (bullet points):**

- 

## 4. Chi-Square Test (Resampling Approach, Fisher's Exact Test, Relevance for Data Science)

Everything so far compared *numeric* outcomes (Sale_Price) across groups. The chi-square test asks a different kind of question: are two **categorical** variables independent of each other, or does knowing one tell you something about the other?

Concrete example: imagine 100 houses cross-tabbed by `Central_Air` (Y/N) and some other categorical trait. If the two were truly independent, each cell's *expected* count would just be `(row total x column total) / grand total`. The chi-square statistic sums up `(observed - expected)^2 / expected` across every cell — big deviations from that independence-implied expectation push the statistic up and the p-value down.

1. Build a contingency table between `Central_Air` and a binned version of `Overall_Qual` (e.g. collapse the 10 tiers into 3 buckets: low / mid / high). Run `scipy.stats.chi2_contingency` on it and interpret the p-value — does having central air look independent of quality tier, or not?
2. **Resampling approach:** instead of trusting the theoretical chi-square distribution, shuffle the `Central_Air` column many times (same idea as the permutation test in notebook 05), recomputing the chi-square statistic from scratch each time to build an empirical null distribution. Compare your resampling-based p-value to the theoretical one from step 1.
3. **Fisher's Exact Test:** find or construct a 2x2 table with small counts (some cell under ~5 — try filtering to a rare `Overall_Qual` tier like `Very_Poor` or `Very_Excellent` crossed with `Central_Air`). Run `scipy.stats.fisher_exact` on it instead of the chi-square test. Why is Fisher's exact the safer choice when counts are this small?
4. **Relevance for Data Science** (reflection only, no code): Ames has 2,930 rows. Plenty of real DS work happens on datasets with millions of rows. As sample size grows huge, what happens to a chi-square test's p-value even for an association that's real but practically meaningless? What should you be looking at *in addition to* the p-value in that setting?


**Your notes (bullet points):**

- 

## 5. Multi-Arm Bandit Algorithm

In your A/B test in notebook 05, you fixed both group sizes ahead of time and only looked at the result at the very end — nobody got "less" of the losing option along the way. A multi-armed bandit instead updates its behavior as data comes in: it keeps sending some traffic to every option (exploring) but increasingly favors whichever option is currently winning (exploiting), so you spend less total time on an option that's already looking bad.

Ames has no live experiment to replay, so simulate one: treat 3 `Overall_Qual` tiers as 3 "arms," where "pulling" an arm means drawing one house's `Sale_Price` at random (with replacement) from that tier — think of it as simulating a business that gets paid `Sale_Price` every time it "serves" that tier to a customer.

1. Pick 3 groups from Ames to act as your 3 arms (e.g. three `Overall_Qual` tiers, or three neighborhoods). Define the reward of pulling an arm as one random draw (with replacement) of `Sale_Price` from that group.
2. Implement an epsilon-greedy bandit: at each round, with probability `epsilon` pick a random arm (explore); otherwise pick whichever arm currently has the highest running-average reward (exploit). Run it for a few thousand rounds, tracking cumulative reward.
3. Compare your epsilon-greedy bandit's cumulative reward against a baseline that just pulls all 3 arms equally, round-robin (the "run a fixed A/B/C test to the end" strategy). Which accumulates more total reward, and why does that mirror the real-world argument for using bandits instead of a fixed-sample A/B test when there's a real business cost to serving the "losing" variant the whole time?


**Your notes (bullet points):**

- 

## 6. Power and Sample Size

Your t-test in notebook 05 found a large price gap between Average and Above_Average quality homes and called it significant. But suppose the *true* population gap were much smaller — say $2,000 instead of ~$27,000. Would the sample size you actually had been enough to reliably catch a gap that small, or would you likely have missed it and wrongly concluded "no difference"? **Power** is the probability your test correctly detects a real effect of a given size, given your sample size and alpha. Flip the question around and you get **sample size planning**: given an effect size you care about and a power level you want, how many observations do you need before you even start collecting data?

1. Using `statsmodels.stats.power.TTestIndPower`, compute the statistical power of the t-test you actually ran in notebook 05 — use your real per-group `n`, the real standardized effect size (Cohen's d = mean difference / pooled standard deviation), and alpha = 0.05.
2. Now compute the required sample size **per group** to detect a much smaller effect (Cohen's d = 0.1, conventionally a "small" effect) at 80% power. How does that number compare to how many houses you actually had in each group?
3. In bullet points: if you were designing a *new* experiment from scratch (not just analyzing Ames after the fact) to detect a $2,000 price difference between two groups, how would a power calculation change what you do *before* collecting any data?


**Your notes (bullet points):**

- 